# 08 Silver Medication Request Clean

## Purpose

This notebook creates the Silver Medication Request table from the Bronze raw MedicationRequest FHIR table.

## What We Are Doing

We will:
1. Read `healthcare_catalog.bronze.medication_request_raw`
2. Extract medication/prescription fields
3. Flatten medication request data
4. Standardize timestamps and patient references
5. Save the clean table into the Silver layer

## Why We Are Doing This

MedicationRequest resources contain prescription orders.

This table is important for:
- medication utilization analytics
- chronic disease treatment analysis
- medication adherence features
- pharmacy analytics
- patient risk scoring
- downstream ML feature engineering

## Expected Final Output

A clean Silver table:

`healthcare_catalog.silver.medication_request_clean`

Expected columns:
- medication_request_id
- patient_id
- encounter_id
- medication_name
- medication_status
- medication_intent
- authored_datetime

## Step 1 — Import PySpark Functions

### What We Are Doing

We are importing PySpark SQL functions.

### Why We Are Doing This

We need Spark functions to extract nested FHIR MedicationRequest fields.

### Expected Output

Spark functions available for this notebook.

In [0]:
from pyspark.sql.functions import *

## Step 2 — Read Bronze Medication Request Table

### What We Are Doing

We are reading the Bronze MedicationRequest table.

### Why We Are Doing This

The Bronze layer contains raw nested FHIR MedicationRequest resources.

### Expected Output

A DataFrame named:

`medication_request_raw_df`

In [0]:
medication_request_raw_df = spark.table(
    "healthcare_catalog.bronze.medication_request_raw"
)

print("Bronze medication_request_raw table loaded successfully.")

Bronze medication_request_raw table loaded successfully.


## Step 3 — Inspect Medication Request Raw Schema

### What We Are Doing

We are printing the MedicationRequest schema.

### Why We Are Doing This

FHIR MedicationRequest resources are nested JSON structures.

We need to identify:
- patient reference
- encounter reference
- medication name
- medication status
- medication intent
- authored date
- dosage instructions

### Expected Output

You should see fields such as:
- resource.id
- resource.subject.reference
- resource.encounter.reference
- resource.medicationCodeableConcept.text
- resource.status
- resource.intent
- resource.authoredOn
- resource.dosageInstruction

In [0]:
medication_request_raw_df.printSchema()

root
 |-- fullUrl: string (nullable = true)
 |-- resourceType: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- abatementDateTime: string (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- activity: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- detail: struct (nullable = true)
 |    |    |    |    |-- code: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |    |    |-- text: string (nullable = true)
 |    |    |    |    |-- location: struct (nullable = true)
 |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |-- statu

## Step 4 — Extract Clean Medication Request Columns

### What We Are Doing

We are extracting medication and prescription-related fields from nested FHIR MedicationRequest resources.

### Why We Are Doing This

MedicationRequest resources contain prescription order information.

This data is important for:
- medication utilization analytics
- chronic disease treatment analytics
- pharmacy analytics
- medication adherence features
- ML feature engineering

### Fields We Will Extract

- medication_request_id
- patient_reference
- encounter_reference
- medication_name
- medication_status
- medication_intent
- authored_datetime
- dosage_text

### Expected Output

A flattened DataFrame named:

`medication_request_clean_df`

In [0]:
medication_request_clean_df = medication_request_raw_df.select(

    col("resource.id").alias("medication_request_id"),

    col("resource.subject.reference").alias("patient_reference"),

    col("resource.encounter.reference").alias("encounter_reference"),

    col("resource.medicationCodeableConcept.text").alias("medication_name"),

    col("resource.status").alias("medication_status"),

    col("resource.intent").alias("medication_intent"),

    col("resource.authoredOn").alias("authored_datetime"),

    col("resource.dosageInstruction")[0]["text"].alias("dosage_text")
)

print("Medication request clean DataFrame created successfully.")

Medication request clean DataFrame created successfully.


## Step 5 — Convert Medication Timestamp

### What We Are Doing

We are converting prescription timestamps into Spark timestamp format.

### Why We Are Doing This

Medication timelines are important for:
- longitudinal patient history
- adherence analysis
- treatment pathway analytics

### Expected Output

Medication timestamps converted successfully.

In [0]:
medication_request_clean_df = medication_request_clean_df.withColumn(
    "authored_datetime",
    to_timestamp(col("authored_datetime"))
)

print("Medication timestamps converted successfully.")

Medication timestamps converted successfully.


## Step 6 — Extract Clean Patient and Encounter IDs

### What We Are Doing

We are extracting clean UUIDs from FHIR references.

### Why We Are Doing This

FHIR references contain:
- urn:uuid:
- ResourceType/ID

We need normalized IDs for joins across healthcare tables.

### Expected Output

New columns:
- patient_id
- encounter_id

In [0]:
medication_request_clean_df = medication_request_clean_df.withColumn(
    "patient_id",

    regexp_extract(
        col("patient_reference"),
        r'urn:uuid:(.*)',
        1
    )
)

medication_request_clean_df = medication_request_clean_df.withColumn(
    "encounter_id",

    regexp_extract(
        col("encounter_reference"),
        r'urn:uuid:(.*)',
        1
    )
)

print("Patient and encounter IDs extracted successfully.")

Patient and encounter IDs extracted successfully.


## Step 7 — Inspect Clean Medication Request Data

### What We Are Doing

We are displaying the flattened medication request table.

### Why We Are Doing This

We need to verify:
- medication extraction works
- timestamps converted correctly
- dosage information extracted correctly
- patient linkage works correctly

### Expected Output

A clean medication request healthcare table.

In [0]:
display(medication_request_clean_df)

medication_request_id,patient_reference,encounter_reference,medication_name,medication_status,medication_intent,authored_datetime,dosage_text,patient_id,encounter_id
0fb6989b-bb5a-dc0f-9163-591969e41198,urn:uuid:fee55adf-498c-5111-2136-e805906a3a74,urn:uuid:f2d77c3d-51e5-9acc-7588-7d7f4bf02b6c,Hydrochlorothiazide 25 MG Oral Tablet,stopped,order,1993-02-15T12:51:27.000Z,null,fee55adf-498c-5111-2136-e805906a3a74,f2d77c3d-51e5-9acc-7588-7d7f4bf02b6c
f7701dc0-fb12-3250-615a-2e0277a287ad,urn:uuid:fee55adf-498c-5111-2136-e805906a3a74,urn:uuid:d8aea876-3378-e2bd-4a36-986628b727f1,amLODIPine 2.5 MG Oral Tablet,stopped,order,1993-03-17T12:51:27.000Z,null,fee55adf-498c-5111-2136-e805906a3a74,d8aea876-3378-e2bd-4a36-986628b727f1
2dca933d-8a15-6ff9-e819-f7c6cd670063,urn:uuid:fee55adf-498c-5111-2136-e805906a3a74,urn:uuid:ddedfee1-cd05-584d-5cdd-78c173eeae09,lisinopril 10 MG Oral Tablet,stopped,order,1993-05-16T12:51:27.000Z,null,fee55adf-498c-5111-2136-e805906a3a74,ddedfee1-cd05-584d-5cdd-78c173eeae09
18cfd9d8-c898-d0eb-f4b4-64a3d66c1e71,urn:uuid:fee55adf-498c-5111-2136-e805906a3a74,urn:uuid:8ed60990-3de1-1a48-3455-0e7512148325,Hydrochlorothiazide 25 MG Oral Tablet,stopped,order,1993-09-20T12:51:27.000Z,null,fee55adf-498c-5111-2136-e805906a3a74,8ed60990-3de1-1a48-3455-0e7512148325
e6a604ac-4b73-f615-84c1-05b9eafc2428,urn:uuid:fee55adf-498c-5111-2136-e805906a3a74,urn:uuid:8ed60990-3de1-1a48-3455-0e7512148325,lisinopril 10 MG Oral Tablet,stopped,order,1993-09-20T12:51:27.000Z,null,fee55adf-498c-5111-2136-e805906a3a74,8ed60990-3de1-1a48-3455-0e7512148325
b2c83dd3-cc95-9322-ef39-b6f45d238e7d,urn:uuid:fee55adf-498c-5111-2136-e805906a3a74,urn:uuid:8ed60990-3de1-1a48-3455-0e7512148325,amLODIPine 2.5 MG Oral Tablet,stopped,order,1993-09-20T12:51:27.000Z,null,fee55adf-498c-5111-2136-e805906a3a74,8ed60990-3de1-1a48-3455-0e7512148325
c7d2e9f6-aa47-7e05-f10d-74c09e8722c3,urn:uuid:fee55adf-498c-5111-2136-e805906a3a74,urn:uuid:0a0d8f16-c177-81e8-ecc8-f58f4e54ff00,Hydrochlorothiazide 25 MG Oral Tablet,stopped,order,1994-02-21T12:51:27.000Z,null,fee55adf-498c-5111-2136-e805906a3a74,0a0d8f16-c177-81e8-ecc8-f58f4e54ff00
e44dbb3f-3be5-7ef5-0635-40e74a451a37,urn:uuid:fee55adf-498c-5111-2136-e805906a3a74,urn:uuid:0a0d8f16-c177-81e8-ecc8-f58f4e54ff00,lisinopril 10 MG Oral Tablet,stopped,order,1994-02-21T12:51:27.000Z,null,fee55adf-498c-5111-2136-e805906a3a74,0a0d8f16-c177-81e8-ecc8-f58f4e54ff00
bbe4acea-8b93-6d57-19cf-10469f4545e3,urn:uuid:fee55adf-498c-5111-2136-e805906a3a74,urn:uuid:0a0d8f16-c177-81e8-ecc8-f58f4e54ff00,amLODIPine 2.5 MG Oral Tablet,stopped,order,1994-02-21T12:51:27.000Z,null,fee55adf-498c-5111-2136-e805906a3a74,0a0d8f16-c177-81e8-ecc8-f58f4e54ff00
52efadb3-440a-13ec-b88b-d70084c5c64a,urn:uuid:fee55adf-498c-5111-2136-e805906a3a74,urn:uuid:3ab928d6-bee1-b454-c642-5b8dcddf3c48,Hydrocortisone 10 MG/ML Topical Cream,active,order,1994-10-30T17:51:27.000Z,Take as needed.,fee55adf-498c-5111-2136-e805906a3a74,3ab928d6-bee1-b454-c642-5b8dcddf3c48


## Step 8 — Check Most Common Medications

### What We Are Doing

We are counting the most prescribed medications.

### Why We Are Doing This

This helps us understand:
- treatment patterns
- chronic disease treatment prevalence
- medication utilization trends

### Expected Output

Top medication frequency table.

In [0]:
display(

    medication_request_clean_df.groupBy(
        "medication_name"
    ).count().orderBy(
        desc("count")
    )

)

medication_name,count
lisinopril 10 MG Oral Tablet,3777
Hydrochlorothiazide 25 MG Oral Tablet,3281
amLODIPine 2.5 MG Oral Tablet,2816
"insulin human, isophane 70 UNT/ML / Regular Insulin, Human 30 UNT/ML Injectable Suspension [Humulin]",1343
120 ACTUAT Fluticasone propionate 0.044 MG/ACTUAT Metered Dose Inhaler,1199
NDA020503 200 ACTUAT Albuterol 0.09 MG/ACTUAT Metered Dose Inhaler,1197
1 ML Epoetin Alfa 4000 UNT/ML Injection [Epogen],1079
24 HR Metformin hydrochloride 500 MG Extended Release Oral Tablet,1030
Simvastatin 10 MG Oral Tablet,904
Digoxin 0.125 MG Oral Tablet,713


## Step 9 — Check Null Values

### What We Are Doing

We are validating missing medication fields.

### Why We Are Doing This

Medication data may contain incomplete prescriptions or missing dosage instructions.

Silver validation is critical before downstream analytics and ML.

### Expected Output

Null count summary.

In [0]:
display(

    medication_request_clean_df.select(

        [
            sum(col(column_name).isNull().cast("int")).alias(column_name)

            for column_name in medication_request_clean_df.columns
        ]

    )

)

medication_request_id,patient_reference,encounter_reference,medication_name,medication_status,medication_intent,authored_datetime,dosage_text,patient_id,encounter_id
0,0,0,354,0,0,0,19936,0,0


## Step 10 — Save Silver Medication Request Table

### What We Are Doing

We are saving the clean medication request table into the Silver layer.

### Why We Are Doing This

The Silver layer stores:
- cleaned
- normalized
- analytics-ready

medication and prescription data.

This table will later support:
- pharmacy analytics
- medication adherence analytics
- chronic disease analytics
- patient risk scoring
- ML feature engineering

### Expected Output

A Delta table:

`healthcare_catalog.silver.medication_request_clean`

In [0]:
medication_request_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.silver.medication_request_clean")

print("Silver medication_request_clean table saved successfully.")

Silver medication_request_clean table saved successfully.


In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.silver
""").show(truncate=False)

+--------+------------------------+-----------+
|database|tableName               |isTemporary|
+--------+------------------------+-----------+
|silver  |condition_clean         |false      |
|silver  |encounter_clean         |false      |
|silver  |medication_request_clean|false      |
|silver  |observation_clean       |false      |
|silver  |patient_clean           |false      |
|silver  |procedure_clean         |false      |
+--------+------------------------+-----------+

